In [2]:
import pickle
import pandas as pd
import numpy as np
import networkx as nx

from scipy.sparse import diags, eye
from scipy.sparse.linalg import spsolve

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# -----------------------------------------
# Load graph
# -----------------------------------------

with open(
    "master_compressed_river_graph.pkl",
    "rb"
) as f:
    G = pickle.load(f)


# -----------------------------------------
# Load hourly data
# -----------------------------------------

hourly = pd.read_csv(
    "usgs_all_stations_hourly.csv"
)


print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

print()
print("Hourly columns:")
print(hourly.columns.tolist())

Graph nodes: 67
Graph edges: 66

Hourly columns:
['station_id', 'time', 'DO concentration', 'Water temperature', 'Specific conductance', 'Turbidity', 'pH', 'Discharge', 'Stage']


In [3]:
TARGET = "pH"

station_values = (
    hourly[
        ["station_id", TARGET]
    ]
    .dropna()
    .groupby("station_id")[TARGET]
    .median()
)

print(
    "Stations with usable",
    TARGET,
    "data:",
    len(station_values)
)

display(
    station_values.head()
)

Stations with usable pH data: 21


station_id
USGS-09070500    8.4
USGS-09071750    8.3
USGS-09085000    8.4
USGS-09085150    7.9
USGS-09085160    8.0
Name: pH, dtype: float64

In [4]:
node_records = []

for node, data in G.nodes(data=True):

    for cluster in data.get(
        "usgs_clusters",
        []
    ):

        member_stations = cluster.get(
            "member_stations",
            []
        )

        values = [
            station_values.loc[station]
            for station in member_stations
            if station in station_values.index
        ]

        if len(values) == 0:
            continue

        node_records.append({
            "node": node,
            "cluster_id":
                cluster.get("cluster_id"),

            "role":
                cluster.get("ml_role"),

            "stations":
                member_stations,

            TARGET:
                float(np.median(values))
        })


node_values = pd.DataFrame(
    node_records
)

display(node_values)

print()
print(
    node_values["role"]
    .value_counts()
)

,node,cluster_id,role,stations,pH
0,97532807,2,train,[USGS-09144250],8.40
1,96496279,3,train,[USGS-09180500],8.30
2,90706215,5,train,[USGS-09163500],8.40
3,67739434,6,train,[USGS-09261000],8.40
4,88786999,8,train,[USGS-09095500],8.40
5,134075067,11,test_partial,[USGS-09365000],8.20
6,93300040,14,train,[USGS-09315000],8.40
7,83027786,15,test_full,[USGS-09071750],8.30
8,81497938,17,test_full,[USGS-09070500],8.40
9,186559442,18,test_partial,[USGS-09510000],8.30



role
train           10
test_partial     5
test_full        4
Name: count, dtype: int64


In [5]:
def build_laplacian(
    G,
    tau_km=100
):

    G_L = nx.Graph()

    G_L.add_nodes_from(
        G.nodes()
    )

    for u, v, data in G.edges(
        data=True
    ):

        distance_km = (
            float(
                data.get(
                    "length_m",
                    0
                )
            )
            / 1000
        )

        weight = np.exp(
            -distance_km / tau_km
        )

        if G_L.has_edge(u, v):

            G_L[u][v][
                "weight"
            ] += weight

        else:

            G_L.add_edge(
                u,
                v,
                weight=weight
            )

    node_order = list(
        G_L.nodes()
    )

    L = nx.laplacian_matrix(
        G_L,
        nodelist=node_order,
        weight="weight"
    ).astype(float)

    return (
        G_L,
        node_order,
        L
    )

In [6]:
G_L, node_order, L = (
    build_laplacian(
        G,
        tau_km=100
    )
)

print("Laplacian shape:", L.shape)
print("Nodes:", len(node_order))

Laplacian shape: (67, 67)
Nodes: 67


In [7]:
def fit_graph_laplacian(
    G,
    node_values,
    target,
    lambda_lap=1.0,
    tau_km=100
):

    G_L, node_order, L = (
        build_laplacian(
            G,
            tau_km=tau_km
        )
    )

    node_index = {
        node: i
        for i, node
        in enumerate(node_order)
    }

    n = len(node_order)

    y = np.zeros(n)

    observed = np.zeros(n)


    # -------------------------------------
    # TRAINING LABELS ONLY
    # -------------------------------------

    training_rows = node_values[
        node_values["role"]
        == "train"
    ]

    for _, row in training_rows.iterrows():

        node = row["node"]

        if node not in node_index:
            continue

        i = node_index[node]

        y[i] = row[target]

        observed[i] = 1


    # -------------------------------------
    # Observation mask
    # -------------------------------------

    M = diags(
        observed
    )


    # -------------------------------------
    # Solve:
    #
    # (M + lambda L) f = M y
    #
    # Small ridge term prevents numerical
    # singularity.
    # -------------------------------------

    epsilon = 1e-8

    A = (
        M
        +
        lambda_lap * L
        +
        epsilon * eye(n)
    )

    b = M @ y

    prediction = spsolve(
        A,
        b
    )


    predictions = {
        node:
            prediction[
                node_index[node]
            ]

        for node in node_order
    }

    return predictions

In [8]:
predictions = fit_graph_laplacian(
    G,
    node_values,
    target="pH",
    lambda_lap=1,
    tau_km=100
)

In [9]:
node = list(G.nodes())[0]

print(
    "Predicted pH:",
    predictions[node]
)

Predicted pH: 8.029403034876466


In [10]:
test = node_values[
    node_values["role"]
    == "test_full"
].copy()

test["predicted_pH"] = (
    test["node"]
    .map(predictions)
)

display(
    test[
        [
            "node",
            "stations",
            "pH",
            "predicted_pH"
        ]
    ]
)

,node,stations,pH,predicted_pH
7,83027786,[USGS-09071750],8.30,8.186677
8,81497938,[USGS-09070500],8.40,8.186677
13,83222751,"[USGS-09085000, USGS-393259107194801]",8.10,8.186678
18,82982643,"[USGS-09085150, USGS-09085160]",7.95,8.186678


In [11]:
y_true = test["pH"]
y_pred = test["predicted_pH"]

mae = mean_absolute_error(
    y_true,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)

r2 = r2_score(
    y_true,
    y_pred
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 0.16250037513497895
RMSE: 0.17455538458037453
R²: -2.7316053411441388e-05


In [12]:
print("Number of test nodes:", len(test))

print("\nActual pH:")
print(test["pH"].describe())

print("\nPredicted pH:")
print(test["predicted_pH"].describe())

print("\nPrediction table:")
display(
    test[
        [
            "node",
            "stations",
            "pH",
            "predicted_pH"
        ]
    ].assign(
        residual=lambda x:
            x["pH"] - x["predicted_pH"]
    )
)

Number of test nodes: 4

Actual pH:
count    4.000000
mean     8.187500
std      0.201556
min      7.950000
25%      8.062500
50%      8.200000
75%      8.325000
max      8.400000
Name: pH, dtype: float64

Predicted pH:
count    4.000000e+00
mean     8.186678e+00
std      5.424181e-07
min      8.186677e+00
25%      8.186677e+00
50%      8.186678e+00
75%      8.186678e+00
max      8.186678e+00
Name: predicted_pH, dtype: float64

Prediction table:


,node,stations,pH,predicted_pH,residual
7,83027786,[USGS-09071750],8.30,8.186677,0.113323
8,81497938,[USGS-09070500],8.40,8.186677,0.213323
13,83222751,"[USGS-09085000, USGS-393259107194801]",8.10,8.186678,-0.086678
18,82982643,"[USGS-09085150, USGS-09085160]",7.95,8.186678,-0.236678


In [13]:
train_mean = node_values.loc[
    node_values["role"] == "train",
    "pH"
].mean()

mean_predictions = np.full(
    len(test),
    train_mean
)

mean_mae = mean_absolute_error(
    test["pH"],
    mean_predictions
)

mean_rmse = np.sqrt(
    mean_squared_error(
        test["pH"],
        mean_predictions
    )
)

mean_r2 = r2_score(
    test["pH"],
    mean_predictions
)

print("Training mean pH:", train_mean)

print("\nMean baseline:")
print("MAE:", mean_mae)
print("RMSE:", mean_rmse)
print("R²:", mean_r2)

print("\nGraph Laplacian:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

Training mean pH: 8.305

Mean baseline:
MAE: 0.1649999999999998
RMSE: 0.21041625412500803
R²: -0.4531282051282002

Graph Laplacian:
MAE: 0.16250037513497895
RMSE: 0.17455538458037453
R²: -2.7316053411441388e-05


In [14]:
training_rows = node_values[
    node_values["role"] == "train"
]

In [15]:
training_rows = node_values[
    node_values["role"] != "test_full"
]

In [16]:
predictions = fit_graph_laplacian(
    G,
    node_values,
    target="pH",
    lambda_lap=1,
    tau_km=100
)

test = node_values[
    node_values["role"] == "test_full"
].copy()

test["predicted_pH"] = (
    test["node"].map(predictions)
)

y_true = test["pH"]
y_pred = test["predicted_pH"]

mae = mean_absolute_error(
    y_true,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)

r2 = r2_score(
    y_true,
    y_pred
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

display(
    test[
        [
            "stations",
            "pH",
            "predicted_pH"
        ]
    ]
)

MAE: 0.16250037513497895
RMSE: 0.17455538458037453
R²: -2.7316053411441388e-05


,stations,pH,predicted_pH
7,[USGS-09071750],8.30,8.186677
8,[USGS-09070500],8.40,8.186677
13,"[USGS-09085000, USGS-393259107194801]",8.10,8.186678
18,"[USGS-09085150, USGS-09085160]",7.95,8.186678


In [17]:
lambda_lap = 1
tau_km = 100

In [18]:
print(
    "Actual test range:",
    test["pH"].min(),
    "to",
    test["pH"].max()
)

print(
    "Predicted test range:",
    test["predicted_pH"].min(),
    "to",
    test["predicted_pH"].max()
)

print(
    "Actual test std:",
    test["pH"].std()
)

print(
    "Predicted test std:",
    test["predicted_pH"].std()
)

Actual test range: 7.95 to 8.4
Predicted test range: 8.18667723102149 to 8.186678472682102
Actual test std: 0.20155644370746398
Predicted test std: 5.424180708776194e-07


In [19]:
lambda_values = [
    0.001,
    0.01,
    0.1,
    1,
    10,
    100
]

tau_values = [
    10,
    25,
    50,
    100,
    200,
    400
]

In [20]:
from scipy.sparse import diags, eye
from scipy.sparse.linalg import spsolve

def fit_graph_laplacian(
    G,
    labels,
    lambda_lap=0.001,
    tau_km=50
):

    # -----------------------------------------
    # Build distance-weighted undirected graph
    # -----------------------------------------

    G_L = nx.Graph()
    G_L.add_nodes_from(G.nodes())

    for u, v, data in G.edges(data=True):

        distance_km = (
            float(data.get("length_m", 0))
            / 1000
        )

        weight = np.exp(
            -distance_km / tau_km
        )

        G_L.add_edge(
            u,
            v,
            weight=weight
        )


    # -----------------------------------------
    # Laplacian
    # -----------------------------------------

    node_order = list(G_L.nodes())

    node_index = {
        node: i
        for i, node in enumerate(node_order)
    }

    L = nx.laplacian_matrix(
        G_L,
        nodelist=node_order,
        weight="weight"
    ).astype(float)


    # -----------------------------------------
    # Observation vector + mask
    # -----------------------------------------

    n = len(node_order)

    y = np.zeros(n)
    mask = np.zeros(n)

    for node, value in labels.items():

        if node not in node_index:
            continue

        i = node_index[node]

        y[i] = value
        mask[i] = 1


    M = diags(mask)


    # -----------------------------------------
    # Solve
    # -----------------------------------------

    A = (
        M
        + lambda_lap * L
        + 1e-10 * eye(n)
    )

    b = M @ y

    f = spsolve(
        A,
        b
    )


    return {
        node: f[node_index[node]]
        for node in node_order
    }

In [21]:
# One row per graph node
node_ph = (
    node_values
    .groupby("node", as_index=False)
    .agg({
        "pH": "median",
        "role": "first",
        "stations": "first"
    })
)

# Final untouched test set
final_test = node_ph[
    node_ph["role"] == "test_full"
].copy()

# Everything else with pH can be used
# for model development / CV
development = node_ph[
    node_ph["role"] != "test_full"
].copy()

print(
    "Development nodes:",
    len(development)
)

print(
    "Final test nodes:",
    len(final_test)
)

Development nodes: 15
Final test nodes: 4


In [22]:
from sklearn.metrics import mean_squared_error

lambda_values = [
    1e-8,
    1e-7,
    1e-6,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    1e-1,
    1
]

tau_values = [
    10,
    25,
    50,
    100,
    200
]

results = []

for tau in tau_values:

    for lam in lambda_values:

        actual = []
        predicted = []

        for held_node in development["node"]:

            train_rows = development[
                development["node"]
                != held_node
            ]

            labels = dict(
                zip(
                    train_rows["node"],
                    train_rows["pH"]
                )
            )

            preds = fit_graph_laplacian(
                G,
                labels,
                lambda_lap=lam,
                tau_km=tau
            )

            actual_value = (
                development.loc[
                    development["node"]
                    == held_node,
                    "pH"
                ].iloc[0]
            )

            actual.append(
                actual_value
            )

            predicted.append(
                preds[held_node]
            )


        rmse = np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        )

        pred_std = np.std(
            predicted
        )

        results.append({
            "lambda": lam,
            "tau_km": tau,
            "cv_rmse": rmse,
            "prediction_std": pred_std
        })


cv_results = (
    pd.DataFrame(results)
    .sort_values("cv_rmse")
)

display(
    cv_results.head(20)
)

,lambda,tau_km,cv_rmse,prediction_std
44,1.000000,200,0.133964,0.087322
35,1.000000,100,0.134828,0.088892
43,0.100000,200,0.138027,0.101410
34,0.100000,100,0.138311,0.100355
42,0.010000,200,0.138790,0.103890
39,0.000010,200,0.138874,0.104273
41,0.001000,200,0.138876,0.104164
40,0.000100,200,0.138881,0.104198
33,0.010000,100,0.138937,0.102266
32,0.001000,100,0.139008,0.102475


In [23]:
lambda_values = [
    0.1,
    0.25,
    0.5,
    1,
    2,
    5,
    10,
    25
]

tau_values = [
    75,
    100,
    150,
    200,
    300,
    400,
    600
]

results = []

for tau in tau_values:

    for lam in lambda_values:

        actual = []
        predicted = []

        for held_node in development["node"]:

            train_rows = development[
                development["node"] != held_node
            ]

            labels = dict(
                zip(
                    train_rows["node"],
                    train_rows["pH"]
                )
            )

            preds = fit_graph_laplacian(
                G,
                labels,
                lambda_lap=lam,
                tau_km=tau
            )

            actual_value = development.loc[
                development["node"] == held_node,
                "pH"
            ].iloc[0]

            actual.append(actual_value)
            predicted.append(
                preds[held_node]
            )

        rmse = np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        )

        mae = mean_absolute_error(
            actual,
            predicted
        )

        results.append({
            "lambda": lam,
            "tau_km": tau,
            "cv_rmse": rmse,
            "cv_mae": mae,
            "actual_std": np.std(actual),
            "prediction_std": np.std(predicted)
        })


refined_results = (
    pd.DataFrame(results)
    .sort_values("cv_rmse")
    .reset_index(drop=True)
)

display(
    refined_results.head(20)
)

,lambda,tau_km,cv_rmse,cv_mae,actual_std,prediction_std
0,25.0,75,0.128744,0.098829,0.131233,0.055203
1,25.0,100,0.128889,0.100979,0.131233,0.048785
2,10.0,150,0.129103,0.099978,0.131233,0.057938
3,10.0,100,0.129183,0.098515,0.131233,0.062615
4,10.0,200,0.129269,0.100796,0.131233,0.055551
5,10.0,300,0.129524,0.102025,0.131233,0.053176
6,25.0,150,0.129651,0.103860,0.131233,0.042329
7,10.0,400,0.129672,0.102670,0.131233,0.052000
8,10.0,600,0.129825,0.103277,0.131233,0.050833
9,5.0,200,0.129856,0.099505,0.131233,0.066411


In [24]:
actual = []
baseline_pred = []

for held_node in development["node"]:

    train_rows = development[
        development["node"] != held_node
    ]

    held_value = development.loc[
        development["node"] == held_node,
        "pH"
    ].iloc[0]

    train_mean = train_rows["pH"].mean()

    actual.append(held_value)
    baseline_pred.append(train_mean)


baseline_rmse = np.sqrt(
    mean_squared_error(
        actual,
        baseline_pred
    )
)

baseline_mae = mean_absolute_error(
    actual,
    baseline_pred
)

baseline_r2 = r2_score(
    actual,
    baseline_pred
)

print("LOOCV mean baseline")
print("RMSE:", baseline_rmse)
print("MAE:", baseline_mae)
print("R²:", baseline_r2)

print()

print("Best graph model so far")
print(
    refined_results.iloc[0]
)

LOOCV mean baseline
RMSE: 0.14060728346449636
MAE: 0.11666666666666678
R²: -0.14795918367346172

Best graph model so far
lambda            25.000000
tau_km            75.000000
cv_rmse            0.128744
cv_mae             0.098829
actual_std         0.131233
prediction_std     0.055203
Name: 0, dtype: float64


In [25]:
lambda_values = [
    5,
    10,
    20,
    25,
    40,
    60,
    100,
    200
]

tau_values = [
    20,
    30,
    40,
    50,
    60,
    75,
    90,
    110,
    150
]

results = []

for tau in tau_values:

    for lam in lambda_values:

        actual = []
        predicted = []

        for held_node in development["node"]:

            train_rows = development[
                development["node"] != held_node
            ]

            labels = dict(
                zip(
                    train_rows["node"],
                    train_rows["pH"]
                )
            )

            preds = fit_graph_laplacian(
                G,
                labels,
                lambda_lap=lam,
                tau_km=tau
            )

            actual_value = development.loc[
                development["node"] == held_node,
                "pH"
            ].iloc[0]

            actual.append(
                actual_value
            )

            predicted.append(
                preds[held_node]
            )

        rmse = np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        )

        mae = mean_absolute_error(
            actual,
            predicted
        )

        r2 = r2_score(
            actual,
            predicted
        )

        results.append({
            "lambda": lam,
            "tau_km": tau,
            "cv_rmse": rmse,
            "cv_mae": mae,
            "cv_r2": r2,
            "actual_std": np.std(actual),
            "prediction_std": np.std(predicted)
        })


final_cv_results = (
    pd.DataFrame(results)
    .sort_values("cv_rmse")
    .reset_index(drop=True)
)

display(
    final_cv_results.head(25)
)

,lambda,tau_km,cv_rmse,cv_mae,cv_r2,actual_std,prediction_std
0,25,75,0.128744,0.098829,0.037580,0.131233,0.055203
1,20,90,0.128746,0.099240,0.037547,0.131233,0.054465
2,25,90,0.128762,0.100062,0.037316,0.131233,0.050965
3,60,60,0.128779,0.099874,0.037059,0.131233,0.048762
4,40,75,0.128799,0.100532,0.036761,0.131233,0.047876
5,40,60,0.128828,0.098599,0.036328,0.131233,0.054866
6,20,110,0.128882,0.100634,0.035522,0.131233,0.050778
7,20,75,0.128895,0.098531,0.035320,0.131233,0.058334
8,100,50,0.128898,0.099764,0.035275,0.131233,0.048484
9,25,110,0.129047,0.101764,0.033045,0.131233,0.046997


In [26]:
best = final_cv_results.iloc[0]

print("Best parameters")
print("---------------------------")
print("lambda:", best["lambda"])
print("tau:", best["tau_km"], "km")
print("RMSE:", best["cv_rmse"])
print("MAE:", best["cv_mae"])
print("R²:", best["cv_r2"])
print()
print("Actual SD:", best["actual_std"])
print("Prediction SD:", best["prediction_std"])

print()
print(
    "Prediction SD / actual SD:",
    best["prediction_std"]
    / best["actual_std"]
)

Best parameters
---------------------------
lambda: 25.0
tau: 75.0 km
RMSE: 0.1287439973504599
MAE: 0.0988287427283223
R²: 0.037579666554989566

Actual SD: 0.1312334645668637
Prediction SD: 0.05520288023661851

Prediction SD / actual SD: 0.42064636804961086


In [27]:
GRAPH_BASELINE = {
    "target": "pH",
    "lambda_lap": 25.0,
    "tau_km": 75.0,
    "cv_rmse": 0.12874399735046288,
    "cv_mae": 0.09882874272832348,
    "cv_r2": 0.037579666554945046,
    "prediction_std_ratio": 0.42064636804960964
}

GRAPH_BASELINE

{'target': 'pH',
 'lambda_lap': 25.0,
 'tau_km': 75.0,
 'cv_rmse': 0.12874399735046288,
 'cv_mae': 0.09882874272832348,
 'cv_r2': 0.037579666554945046,
 'prediction_std_ratio': 0.42064636804960964}

In [28]:
print(hourly.columns.tolist())

print("\nNon-null counts:")
print(
    hourly.notna()
    .sum()
    .sort_values(ascending=False)
)

print("\nFirst 5 rows:")
display(hourly.head())

['station_id', 'time', 'DO concentration', 'Water temperature', 'Specific conductance', 'Turbidity', 'pH', 'Discharge', 'Stage']

Non-null counts:
station_id              234095
time                    234095
Water temperature       157136
Specific conductance    156329
pH                      127454
Discharge               127119
DO concentration        120688
Turbidity                95106
Stage                     8761
dtype: int64

First 5 rows:


,station_id,time,DO concentration,Water temperature,Specific conductance,Turbidity,pH,Discharge,Stage
0,USGS-09070500,2025-10-02 04:00:00+00:00,8.0,14.20,382.0,4.20,8.60,1350.0,NaN
1,USGS-09070500,2025-10-02 05:00:00+00:00,7.9,14.00,381.0,4.40,8.50,1350.0,NaN
2,USGS-09070500,2025-10-02 06:00:00+00:00,7.9,13.75,381.0,4.40,8.50,1350.0,NaN
3,USGS-09070500,2025-10-02 07:00:00+00:00,7.9,13.45,380.5,4.40,8.45,1345.0,NaN
4,USGS-09070500,2025-10-02 08:00:00+00:00,7.9,13.15,380.5,4.45,8.40,1340.0,NaN


In [29]:
import numpy as np
import pandas as pd
import networkx as nx

from scipy.sparse import (
    csr_matrix,
    diags,
    eye,
    bmat
)

from scipy.sparse.linalg import spsolve

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


TARGET = "pH"

FEATURES = [
    "Water temperature",
    "Specific conductance",
    "DO concentration",
    "Turbidity"
]


# Make sure everything is numeric
for col in [TARGET] + FEATURES:
    hourly[col] = pd.to_numeric(
        hourly[col],
        errors="coerce"
    )

hourly["station_id"] = (
    hourly["station_id"]
    .astype(str)
)

print("Target:", TARGET)
print("Features:", FEATURES)

Target: pH
Features: ['Water temperature', 'Specific conductance', 'DO concentration', 'Turbidity']


In [30]:
# ==========================================
# Station-level medians
# ==========================================

station_summary = (
    hourly
    .groupby("station_id")[
        [TARGET] + FEATURES
    ]
    .median()
)


# ==========================================
# Move station data onto graph nodes
# ==========================================

rows = []

for node, data in G.nodes(data=True):

    for cluster in data.get(
        "usgs_clusters",
        []
    ):

        members = [
            str(x)
            for x in cluster.get(
                "member_stations",
                []
            )
        ]

        present = [
            station
            for station in members
            if station
            in station_summary.index
        ]

        if len(present) == 0:
            continue

        station_data = (
            station_summary.loc[present]
        )

        target_value = (
            station_data[TARGET]
            .median()
        )

        # Can't use this node for pH modeling
        # if it has no pH
        if pd.isna(target_value):
            continue

        row = {
            "node": node,
            "role": cluster.get(
                "ml_role"
            ),
            "stations": members,
            TARGET: target_value
        }

        for feature in FEATURES:

            row[feature] = (
                station_data[feature]
                .median()
            )

        rows.append(row)


node_table = pd.DataFrame(rows)


# ==========================================
# Check for duplicate effective nodes
# ==========================================

print(
    "Rows:",
    len(node_table)
)

print(
    "Unique graph nodes:",
    node_table["node"].nunique()
)

print()

print(
    node_table["role"]
    .value_counts()
)

print()

print("Missing predictors:")
print(
    node_table[FEATURES]
    .isna()
    .sum()
)

display(
    node_table.head()
)

Rows: 19
Unique graph nodes: 19

role
train           10
test_partial     5
test_full        4
Name: count, dtype: int64

Missing predictors:
Water temperature       0
Specific conductance    0
DO concentration        3
Turbidity               2
dtype: int64


,node,role,stations,pH,Water temperature,Specific conductance,DO concentration,Turbidity
0,97532807,train,[USGS-09144250],8.4,8.30,627.5,10.3,11.450
1,96496279,train,[USGS-09180500],8.3,14.05,1190.0,8.8,28.475
2,90706215,train,[USGS-09163500],8.4,9.20,1270.0,10.1,33.450
3,67739434,train,[USGS-09261000],8.4,12.10,634.0,9.0,14.500
4,88786999,train,[USGS-09095500],8.4,11.45,1140.0,10.2,6.250


In [31]:
USE_PARTIAL_AS_TRAIN = True


# Final untouched station test set
final_test = node_table[
    node_table["role"]
    == "test_full"
].copy()


if USE_PARTIAL_AS_TRAIN:

    # Everything with pH except final test
    development = node_table[
        node_table["role"]
        != "test_full"
    ].copy()

else:

    # Strict original split
    development = node_table[
        node_table["role"]
        == "train"
    ].copy()


print(
    "Development nodes:",
    len(development)
)

print(
    "Final test nodes:",
    len(final_test)
)

print()

print(
    development["role"]
    .value_counts()
)

Development nodes: 15
Final test nodes: 4

role
train           10
test_partial     5
Name: count, dtype: int64


In [32]:
def build_laplacian(
    G,
    tau_km
):

    G_L = nx.Graph()

    G_L.add_nodes_from(
        G.nodes()
    )


    for u, v, data in G.edges(
        data=True
    ):

        distance_km = (
            float(
                data.get(
                    "length_m",
                    0
                )
            )
            / 1000
        )

        weight = np.exp(
            -distance_km
            / tau_km
        )


        if G_L.has_edge(u, v):

            G_L[u][v][
                "weight"
            ] += weight

        else:

            G_L.add_edge(
                u,
                v,
                weight=weight
            )


    node_order = list(
        G_L.nodes()
    )

    L = nx.laplacian_matrix(
        G_L,
        nodelist=node_order,
        weight="weight"
    ).astype(float)


    return (
        G_L,
        node_order,
        L
    )

In [33]:
def fit_graph_laplacian_regression(
    node_table,
    train_nodes,
    node_order,
    L,
    target,
    features,
    lambda_lap=25,
    lambda_beta=1
):

    table = (
        node_table
        .set_index("node")
    )

    node_index = {
        node: i
        for i, node
        in enumerate(node_order)
    }

    n = len(node_order)
    p = len(features)


    # ======================================
    # DESIGN MATRIX
    # ======================================

    # +1 for intercept
    X = np.ones(
        (
            n,
            p + 1
        )
    )

    preprocessing = {}


    for j, feature in enumerate(
        features,
        start=1
    ):

        train_values = []

        for node in train_nodes:

            if node not in table.index:
                continue

            value = table.loc[
                node,
                feature
            ]

            train_values.append(
                value
            )


        train_values = pd.Series(
            train_values,
            dtype=float
        )


        # ----------------------------------
        # Training-only median imputation
        # ----------------------------------

        median = train_values.median()

        if pd.isna(median):
            median = 0.0


        imputed_train = (
            train_values
            .fillna(median)
        )


        # ----------------------------------
        # Training-only scaling
        # ----------------------------------

        mean = imputed_train.mean()

        std = imputed_train.std(
            ddof=0
        )

        if (
            pd.isna(std)
            or std < 1e-12
        ):
            std = 1.0


        preprocessing[feature] = {
            "median": median,
            "mean": mean,
            "std": std
        }


        # ----------------------------------
        # Fill all graph nodes
        # ----------------------------------

        for node in node_order:

            i = node_index[node]

            if node in table.index:

                value = table.loc[
                    node,
                    feature
                ]

            else:

                value = np.nan


            if pd.isna(value):
                value = median


            X[i, j] = (
                (value - mean)
                / std
            )


    # ======================================
    # TARGET + OBSERVATION MASK
    # ======================================

    y = np.zeros(n)

    mask = np.zeros(n)


    for node in train_nodes:

        if node not in table.index:
            continue

        value = table.loc[
            node,
            target
        ]

        if pd.isna(value):
            continue

        i = node_index[node]

        y[i] = value

        mask[i] = 1.0


    M = diags(mask)

    X_sparse = csr_matrix(X)


    # ======================================
    # RIDGE PENALTY FOR BETA
    #
    # Intercept gets penalty 0.
    # ======================================

    beta_penalty = np.ones(
        p + 1
    )

    beta_penalty[0] = 0

    R = diags(
        beta_penalty
    )


    # ======================================
    # BLOCK NORMAL EQUATIONS
    # ======================================

    A11 = (
        X_sparse.T
        @ M
        @ X_sparse
        +
        lambda_beta * R
    )

    A12 = (
        X_sparse.T
        @ M
    )

    A21 = (
        M
        @ X_sparse
    )

    A22 = (
        M
        +
        lambda_lap * L
        +
        1e-8 * eye(n)
    )


    A = bmat(
        [
            [A11, A12],
            [A21, A22]
        ],
        format="csr"
    )


    My = M @ y

    rhs_beta = (
        X_sparse.T
        @ My
    )

    rhs = np.concatenate([
        np.asarray(
            rhs_beta
        ).ravel(),

        np.asarray(
            My
        ).ravel()
    ])


    # ======================================
    # SOLVE
    # ======================================

    solution = spsolve(
        A,
        rhs
    )


    beta = solution[
        :p + 1
    ]

    g = solution[
        p + 1:
    ]


    predictions = (
        X @ beta
        + g
    )


    prediction_dict = {
        node:
            predictions[
                node_index[node]
            ]

        for node
        in node_order
    }


    graph_effect_dict = {
        node:
            g[
                node_index[node]
            ]

        for node
        in node_order
    }


    coefficient_names = (
        ["Intercept"]
        + features
    )

    coefficients = dict(
        zip(
            coefficient_names,
            beta
        )
    )


    return {
        "predictions":
            prediction_dict,

        "graph_effect":
            graph_effect_dict,

        "coefficients":
            coefficients,

        "preprocessing":
            preprocessing
    }

In [34]:
lambda_lap_values = [
    5,
    10,
    25,
    50,
    100
]

tau_values = [
    25,
    50,
    75,
    100,
    150
]

lambda_beta_values = [
    0.01,
    0.1,
    1,
    10,
    100
]


results = []


for tau in tau_values:

    # Only need to build L once
    # for each tau
    G_L, node_order, L = (
        build_laplacian(
            G,
            tau_km=tau
        )
    )


    for lambda_lap in lambda_lap_values:

        for lambda_beta in lambda_beta_values:

            actual = []
            predicted = []


            for held_node in development[
                "node"
            ]:

                train_nodes = (
                    development.loc[
                        development["node"]
                        != held_node,
                        "node"
                    ]
                    .tolist()
                )


                model = (
                    fit_graph_laplacian_regression(
                        node_table=
                            node_table,

                        train_nodes=
                            train_nodes,

                        node_order=
                            node_order,

                        L=L,

                        target=
                            TARGET,

                        features=
                            FEATURES,

                        lambda_lap=
                            lambda_lap,

                        lambda_beta=
                            lambda_beta
                    )
                )


                actual_value = (
                    development.loc[
                        development["node"]
                        == held_node,
                        TARGET
                    ]
                    .iloc[0]
                )


                predicted_value = (
                    model[
                        "predictions"
                    ][held_node]
                )


                actual.append(
                    actual_value
                )

                predicted.append(
                    predicted_value
                )


            rmse = np.sqrt(
                mean_squared_error(
                    actual,
                    predicted
                )
            )


            mae = mean_absolute_error(
                actual,
                predicted
            )


            r2 = r2_score(
                actual,
                predicted
            )


            results.append({
                "lambda_lap":
                    lambda_lap,

                "tau_km":
                    tau,

                "lambda_beta":
                    lambda_beta,

                "cv_rmse":
                    rmse,

                "cv_mae":
                    mae,

                "cv_r2":
                    r2,

                "actual_std":
                    np.std(
                        actual
                    ),

                "prediction_std":
                    np.std(
                        predicted
                    )
            })


glr_results = (
    pd.DataFrame(results)
    .sort_values(
        "cv_rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    glr_results.head(20)
)

,lambda_lap,tau_km,lambda_beta,cv_rmse,cv_mae,cv_r2,actual_std,prediction_std
0,50,50,0.10,0.089258,0.069668,0.537398,0.131233,0.133315
1,100,50,0.10,0.089370,0.070864,0.536236,0.131233,0.129933
2,100,50,0.01,0.089491,0.068959,0.534979,0.131233,0.133117
3,50,50,0.01,0.089491,0.069026,0.534979,0.131233,0.137027
4,25,50,0.10,0.090022,0.072292,0.529443,0.131233,0.135674
5,25,50,0.01,0.090339,0.071153,0.526125,0.131233,0.140236
6,25,75,0.10,0.091259,0.071716,0.516429,0.131233,0.127418
7,10,75,0.10,0.091337,0.071031,0.515597,0.131233,0.130935
8,25,75,0.01,0.091649,0.070453,0.512281,0.131233,0.130732
9,5,75,0.10,0.091863,0.072371,0.510006,0.131233,0.132132


In [35]:
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

lambda_beta_values = [
    0.001,
    0.01,
    0.1,
    1,
    10,
    100
]

linear_results = []

for lambda_beta in lambda_beta_values:

    actual = []
    predicted = []

    for held_node in development["node"]:

        train = development[
            development["node"] != held_node
        ].copy()

        held = development[
            development["node"] == held_node
        ].copy()

        # -----------------------------------
        # Training-only imputation
        # -----------------------------------

        medians = (
            train[FEATURES]
            .median()
        )

        X_train = (
            train[FEATURES]
            .fillna(medians)
        )

        X_test = (
            held[FEATURES]
            .fillna(medians)
        )


        # -----------------------------------
        # Training-only standardization
        # -----------------------------------

        means = X_train.mean()
        stds = X_train.std(ddof=0)

        stds = stds.replace(
            0,
            1
        )

        X_train = (
            X_train - means
        ) / stds

        X_test = (
            X_test - means
        ) / stds


        # -----------------------------------
        # Ridge regression
        # -----------------------------------

        model = Ridge(
            alpha=lambda_beta
        )

        model.fit(
            X_train,
            train[TARGET]
        )

        pred = model.predict(
            X_test
        )[0]

        actual.append(
            held[TARGET].iloc[0]
        )

        predicted.append(
            pred
        )


    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    mae = mean_absolute_error(
        actual,
        predicted
    )

    r2 = r2_score(
        actual,
        predicted
    )

    linear_results.append({
        "lambda_beta":
            lambda_beta,

        "cv_rmse":
            rmse,

        "cv_mae":
            mae,

        "cv_r2":
            r2,

        "actual_std":
            np.std(actual),

        "prediction_std":
            np.std(predicted)
    })


linear_results = (
    pd.DataFrame(
        linear_results
    )
    .sort_values(
        "cv_rmse"
    )
    .reset_index(
        drop=True
    )
)

display(
    linear_results
)

,lambda_beta,cv_rmse,cv_mae,cv_r2,actual_std,prediction_std
0,1.000,0.105509,0.087296,0.353622,0.131233,0.107654
1,0.100,0.108219,0.088056,0.319984,0.131233,0.121761
2,0.010,0.108829,0.088143,0.312302,0.131233,0.123647
3,0.001,0.108895,0.088152,0.311463,0.131233,0.123842
4,10.000,0.112568,0.091660,0.264239,0.131233,0.061286
5,100.000,0.132770,0.109270,-0.023549,0.131233,0.011833


In [36]:
# ==================================================
# FINAL COMBINED MODEL
# ==================================================

BEST_TAU = 50
BEST_LAMBDA_LAP = 50
BEST_LAMBDA_BETA = 0.1


# Build Laplacian using selected tau
G_L, node_order, L = build_laplacian(
    G,
    tau_km=BEST_TAU
)


# All development nodes are now used for fitting
train_nodes = (
    development["node"]
    .tolist()
)


final_model = fit_graph_laplacian_regression(
    node_table=node_table,
    train_nodes=train_nodes,
    node_order=node_order,
    L=L,
    target=TARGET,
    features=FEATURES,
    lambda_lap=BEST_LAMBDA_LAP,
    lambda_beta=BEST_LAMBDA_BETA
)


# ==================================================
# PREDICT HELD-OUT TEST STATIONS
# ==================================================

final_test_eval = final_test.copy()

final_test_eval["predicted_pH"] = (
    final_test_eval["node"]
    .map(
        final_model["predictions"]
    )
)

final_test_eval["residual"] = (
    final_test_eval["pH"]
    -
    final_test_eval["predicted_pH"]
)


# ==================================================
# METRICS
# ==================================================

y_true = final_test_eval["pH"]
y_pred = final_test_eval["predicted_pH"]

final_mae = mean_absolute_error(
    y_true,
    y_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)

final_r2 = r2_score(
    y_true,
    y_pred
)


print("FINAL HELD-OUT RESULTS")
print("---------------------------")
print("MAE:", final_mae)
print("RMSE:", final_rmse)
print("R²:", final_r2)

print()
print("Actual SD:", y_true.std())
print("Prediction SD:", y_pred.std())

print()
display(
    final_test_eval[
        [
            "stations",
            "pH",
            "predicted_pH",
            "residual"
        ]
    ]
)

FINAL HELD-OUT RESULTS
---------------------------
MAE: 0.15240216612552593
RMSE: 0.19755609098742202
R²: -0.2809324007788463

Actual SD: 0.20155644370746398
Prediction SD: 0.06246216977291922



,stations,pH,predicted_pH,residual
7,[USGS-09071750],8.30,8.244129,0.055871
8,[USGS-09070500],8.40,8.388499,0.011501
13,"[USGS-09085000, USGS-393259107194801]",8.10,8.317039,-0.217039
18,"[USGS-09085150, USGS-09085160]",7.95,8.275198,-0.325198


In [37]:
coefficients = (
    pd.Series(
        final_model["coefficients"],
        name="coefficient"
    )
)

print("MODEL COEFFICIENTS")
print("---------------------------")

display(
    coefficients
    .to_frame()
)

MODEL COEFFICIENTS
---------------------------


,coefficient
Intercept,8.281900
Water temperature,0.054419
Specific conductance,0.011250
DO concentration,0.134385
Turbidity,0.078634


In [38]:
feature_coefficients = (
    coefficients
    .drop("Intercept")
    .to_frame()
)

feature_coefficients[
    "absolute_effect"
] = (
    feature_coefficients[
        "coefficient"
    ].abs()
)

display(
    feature_coefficients
    .sort_values(
        "absolute_effect",
        ascending=False
    )
)

,coefficient,absolute_effect
DO concentration,0.134385,0.134385
Turbidity,0.078634,0.078634
Water temperature,0.054419,0.054419
Specific conductance,0.011250,0.011250


In [39]:
graph_effect_rows = []

for node in final_test_eval["node"]:

    observed = final_test_eval.loc[
        final_test_eval["node"] == node,
        "pH"
    ].iloc[0]

    predicted = (
        final_model[
            "predictions"
        ][node]
    )

    graph_effect = (
        final_model[
            "graph_effect"
        ][node]
    )

    linear_part = (
        predicted
        - graph_effect
    )

    graph_effect_rows.append({
        "node": node,
        "observed_pH": observed,
        "linear_component": linear_part,
        "graph_component": graph_effect,
        "final_prediction": predicted
    })


component_table = pd.DataFrame(
    graph_effect_rows
)

display(component_table)

,node,observed_pH,linear_component,graph_component,final_prediction
0,83027786,8.30,8.244749,-0.00062,8.244129
1,81497938,8.40,8.389119,-0.00062,8.388499
2,83222751,8.10,8.317659,-0.00062,8.317039
3,82982643,7.95,8.275818,-0.00062,8.275198


In [40]:
from scipy.sparse import (
    csr_matrix,
    diags,
    bmat
)

from scipy.sparse.linalg import spsolve


def fit_graph_laplacian_regression(
    node_table,
    train_nodes,
    node_order,
    L,
    target,
    features,
    lambda_lap=50,
    lambda_beta=0.1
):

    table = node_table.set_index("node")

    node_index = {
        node: i
        for i, node in enumerate(node_order)
    }

    n = len(node_order)
    p = len(features)


    # ==================================================
    # DESIGN MATRIX
    # ==================================================

    X = np.ones(
        (n, p + 1)
    )

    preprocessing = {}


    for j, feature in enumerate(
        features,
        start=1
    ):

        train_values = pd.Series(
            [
                table.loc[node, feature]
                for node in train_nodes
                if node in table.index
            ],
            dtype=float
        )

        # Training-only imputation
        median = train_values.median()

        if pd.isna(median):
            median = 0.0

        imputed = train_values.fillna(
            median
        )

        # Training-only scaling
        mean = imputed.mean()

        std = imputed.std(
            ddof=0
        )

        if (
            pd.isna(std)
            or std < 1e-12
        ):
            std = 1.0


        preprocessing[feature] = {
            "median": median,
            "mean": mean,
            "std": std
        }


        for node in node_order:

            i = node_index[node]

            if node in table.index:
                value = table.loc[
                    node,
                    feature
                ]
            else:
                value = np.nan

            if pd.isna(value):
                value = median

            X[i, j] = (
                value - mean
            ) / std


    # ==================================================
    # TARGET + OBSERVATION MASK
    # ==================================================

    y = np.zeros(n)
    mask = np.zeros(n)

    for node in train_nodes:

        if node not in table.index:
            continue

        value = table.loc[
            node,
            target
        ]

        if pd.isna(value):
            continue

        i = node_index[node]

        y[i] = value
        mask[i] = 1.0


    M = diags(mask)

    Xs = csr_matrix(X)


    # ==================================================
    # BETA RIDGE PENALTY
    #
    # Do not penalize intercept
    # ==================================================

    beta_penalty = np.ones(
        p + 1
    )

    beta_penalty[0] = 0

    R = diags(
        beta_penalty
    )


    # ==================================================
    # NORMAL EQUATIONS
    # ==================================================

    A11 = (
        Xs.T @ M @ Xs
        +
        lambda_beta * R
    )

    A12 = Xs.T @ M

    A21 = M @ Xs

    A22 = (
        M
        +
        lambda_lap * L
    )


    # ==================================================
    # SUM-TO-ZERO CONSTRAINT FOR GRAPH EFFECT
    #
    # sum(g) = 0
    # ==================================================

    ones_col = csr_matrix(
        np.ones((n, 1))
    )

    ones_row = csr_matrix(
        np.ones((1, n))
    )

    zero_beta_col = csr_matrix(
        (p + 1, 1)
    )

    zero_beta_row = csr_matrix(
        (1, p + 1)
    )

    zero_scalar = csr_matrix(
        (1, 1)
    )


    A = bmat(
        [
            [
                A11,
                A12,
                zero_beta_col
            ],
            [
                A21,
                A22,
                ones_col
            ],
            [
                zero_beta_row,
                ones_row,
                zero_scalar
            ]
        ],
        format="csr"
    )


    My = M @ y

    rhs_beta = (
        Xs.T @ My
    )

    rhs = np.concatenate([
        np.asarray(
            rhs_beta
        ).ravel(),

        np.asarray(
            My
        ).ravel(),

        [0.0]
    ])


    # ==================================================
    # SOLVE
    # ==================================================

    solution = spsolve(
        A,
        rhs
    )


    beta = solution[
        :p + 1
    ]

    g = solution[
        p + 1:
        p + 1 + n
    ]


    predictions = (
        X @ beta
        + g
    )


    coefficient_names = (
        ["Intercept"]
        + features
    )


    return {

        "predictions": {
            node:
                predictions[
                    node_index[node]
                ]
            for node in node_order
        },

        "graph_effect": {
            node:
                g[
                    node_index[node]
                ]
            for node in node_order
        },

        "coefficients":
            dict(
                zip(
                    coefficient_names,
                    beta
                )
            ),

        "preprocessing":
            preprocessing
    }

In [41]:
graph_effects = pd.Series(
    final_model["graph_effect"]
)

print(
    graph_effects.describe()
)

print(
    "Graph effect std:",
    graph_effects.std()
)

print(
    "Graph effect min:",
    graph_effects.min()
)

print(
    "Graph effect max:",
    graph_effects.max()
)

print(
    "Graph effect sum:",
    graph_effects.sum()
)

count    6.700000e+01
mean     5.905864e-09
std      7.736377e-02
min     -1.387414e-01
25%     -2.708334e-02
50%      3.302992e-03
75%      1.547596e-02
max      1.479310e-01
dtype: float64
Graph effect std: 0.07736376783216083
Graph effect min: -0.13874141658610875
Graph effect max: 0.1479309674695512
Graph effect sum: 3.956929177329968e-07


In [42]:
G_distance = G.to_undirected(
    as_view=True
)

development_nodes = set(
    development["node"]
)

coverage_rows = []

for node in final_test["node"]:

    distances = nx.single_source_dijkstra_path_length(
        G_distance,
        node,
        weight="length_m"
    )

    nearby_training = {
        n: d
        for n, d in distances.items()
        if n in development_nodes
    }

    if nearby_training:

        nearest_node = min(
            nearby_training,
            key=nearby_training.get
        )

        distance_km = (
            nearby_training[
                nearest_node
            ]
            / 1000
        )

    else:

        nearest_node = None
        distance_km = np.nan


    coverage_rows.append({
        "test_node": node,
        "nearest_training_node":
            nearest_node,
        "nearest_training_distance_km":
            distance_km
    })


test_coverage = pd.DataFrame(
    coverage_rows
)

display(test_coverage)

,test_node,nearest_training_node,nearest_training_distance_km
0,83027786,84197086,75.046664
1,81497938,84197086,100.692823
2,83222751,84197086,69.781019
3,82982643,84197086,61.423837


In [43]:
nearest_train = 84197086

path_rows = []

for test_node in final_test["node"]:

    path = nx.shortest_path(
        G_distance,
        test_node,
        nearest_train,
        weight="length_m"
    )

    distance_km = nx.shortest_path_length(
        G_distance,
        test_node,
        nearest_train,
        weight="length_m"
    ) / 1000

    junctions_on_path = [
        node
        for node in path
        if (
            G.in_degree(node)
            + G.out_degree(node)
            >= 3
        )
    ]

    path_rows.append({
        "test_node": test_node,
        "distance_km": distance_km,
        "path_nodes": len(path),
        "junctions_on_path": len(junctions_on_path),
        "junction_nodes": junctions_on_path
    })


path_report = pd.DataFrame(
    path_rows
)

display(path_report)

,test_node,distance_km,path_nodes,junctions_on_path,junction_nodes
0,83027786,75.046664,6,0,[]
1,81497938,100.692823,7,0,[]
2,83222751,69.781019,5,0,[]
3,82982643,61.423837,3,0,[]


In [44]:
all_wq_nodes = set(
    node_table["node"]
)

neighbor_rows = []

for test_node in final_test["node"]:

    distances = nx.single_source_dijkstra_path_length(
        G_distance,
        test_node,
        weight="length_m"
    )

    nearby_wq = sorted(
        [
            (node, distance / 1000)
            for node, distance in distances.items()
            if (
                node in all_wq_nodes
                and node != test_node
            )
        ],
        key=lambda x: x[1]
    )

    for rank, (node, distance) in enumerate(
        nearby_wq[:5],
        start=1
    ):

        role = node_table.loc[
            node_table["node"] == node,
            "role"
        ]

        role = (
            role.iloc[0]
            if len(role)
            else None
        )

        neighbor_rows.append({
            "test_node": test_node,
            "rank": rank,
            "neighbor_node": node,
            "neighbor_role": role,
            "distance_km": distance
        })


nearest_wq_report = pd.DataFrame(
    neighbor_rows
)

display(
    nearest_wq_report
)

,test_node,rank,neighbor_node,neighbor_role,distance_km
0,83027786,1,83222751,test_full,5.265645
1,83027786,2,82982643,test_full,13.622826
2,83027786,3,81497938,test_full,25.646160
3,83027786,4,84197086,train,75.046664
4,83027786,5,88786999,train,128.002616
5,81497938,1,83027786,test_full,25.646160
6,81497938,2,83222751,test_full,30.911804
7,81497938,3,82982643,test_full,39.268986
8,81497938,4,84197086,train,100.692823
9,81497938,5,88786999,train,153.648776


In [45]:
# ============================================================
# PRECOMPUTE RIVER DISTANCES BETWEEN DEVELOPMENT STATIONS
# ============================================================

G_distance = G.to_undirected(
    as_view=True
)

development_nodes = (
    development["node"]
    .tolist()
)

dev_distances = {}

for node in development_nodes:

    distances = nx.single_source_dijkstra_path_length(
        G_distance,
        node,
        weight="length_m"
    )

    dev_distances[node] = {
        other: distances[other] / 1000
        for other in development_nodes
        if other in distances
    }


print(
    "Development nodes:",
    len(development_nodes)
)

Development nodes: 15


In [46]:
BUFFER_KM = 50

lambda_lap_values = [
    10,
    25,
    50,
    75,
    100
]

tau_values = [
    25,
    50,
    75,
    100,
    150
]

lambda_beta_values = [
    0.01,
    0.1,
    1,
    10
]


buffered_results = []


for tau in tau_values:

    G_L, node_order, L = build_laplacian(
        G,
        tau_km=tau
    )

    for lambda_lap in lambda_lap_values:

        for lambda_beta in lambda_beta_values:

            actual = []
            predicted = []
            train_sizes = []


            for held_node in development_nodes:

                # ----------------------------------------
                # REMOVE HELD NODE + LOCAL NEIGHBORS
                # ----------------------------------------

                train_nodes = [
                    node
                    for node in development_nodes

                    if (
                        node != held_node

                        and

                        dev_distances[
                            held_node
                        ].get(
                            node,
                            np.inf
                        ) > BUFFER_KM
                    )
                ]


                # Skip if buffer leaves too little data
                if len(train_nodes) < 5:
                    continue


                model = fit_graph_laplacian_regression(
                    node_table=node_table,
                    train_nodes=train_nodes,
                    node_order=node_order,
                    L=L,
                    target=TARGET,
                    features=FEATURES,
                    lambda_lap=lambda_lap,
                    lambda_beta=lambda_beta
                )


                observed = (
                    development.loc[
                        development["node"]
                        == held_node,
                        TARGET
                    ]
                    .iloc[0]
                )


                pred = model[
                    "predictions"
                ][held_node]


                actual.append(
                    observed
                )

                predicted.append(
                    pred
                )

                train_sizes.append(
                    len(train_nodes)
                )


            if len(actual) < 3:
                continue


            rmse = np.sqrt(
                mean_squared_error(
                    actual,
                    predicted
                )
            )

            mae = mean_absolute_error(
                actual,
                predicted
            )

            r2 = r2_score(
                actual,
                predicted
            )


            buffered_results.append({
                "buffer_km":
                    BUFFER_KM,

                "lambda_lap":
                    lambda_lap,

                "tau_km":
                    tau,

                "lambda_beta":
                    lambda_beta,

                "cv_rmse":
                    rmse,

                "cv_mae":
                    mae,

                "cv_r2":
                    r2,

                "actual_std":
                    np.std(actual),

                "prediction_std":
                    np.std(predicted),

                "mean_training_nodes":
                    np.mean(train_sizes),

                "n_validation_predictions":
                    len(actual)
            })


buffered_results = (
    pd.DataFrame(
        buffered_results
    )
    .sort_values(
        "cv_rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    buffered_results.head(20)
)

,buffer_km,lambda_lap,tau_km,lambda_beta,cv_rmse,cv_mae,cv_r2,actual_std,prediction_std,mean_training_nodes,n_validation_predictions
0,50,75,50,0.10,0.089210,0.069949,0.537894,0.131233,0.131434,14.0,15
1,50,50,50,0.10,0.089258,0.069668,0.537398,0.131233,0.133315,14.0,15
2,50,100,50,0.10,0.089370,0.070864,0.536236,0.131233,0.129933,14.0,15
3,50,75,50,0.01,0.089385,0.068507,0.536085,0.131233,0.134815,14.0,15
4,50,100,50,0.01,0.089491,0.068959,0.534979,0.131233,0.133117,14.0,15
5,50,50,50,0.01,0.089491,0.069026,0.534979,0.131233,0.137027,14.0,15
6,50,25,50,0.10,0.090022,0.072292,0.529443,0.131233,0.135674,14.0,15
7,50,25,50,0.01,0.090339,0.071153,0.526125,0.131233,0.140236,14.0,15
8,50,25,75,0.10,0.091259,0.071716,0.516429,0.131233,0.127418,14.0,15
9,50,10,75,0.10,0.091337,0.071031,0.515597,0.131233,0.130935,14.0,15


In [47]:
BUFFER_VALUES = [
    75,
    100,
    125,
    150
]

lambda_lap_values = [
    25,
    50,
    75,
    100
]

tau_values = [
    25,
    50,
    75,
    100
]

lambda_beta_values = [
    0.01,
    0.1,
    1
]


all_buffer_results = []


for BUFFER_KM in BUFFER_VALUES:

    print(
        "\nRunning buffer:",
        BUFFER_KM,
        "km"
    )

    for tau in tau_values:

        G_L, node_order, L = (
            build_laplacian(
                G,
                tau_km=tau
            )
        )

        for lambda_lap in lambda_lap_values:

            for lambda_beta in lambda_beta_values:

                actual = []
                predicted = []

                train_sizes = []
                removed_counts = []


                for held_node in development_nodes:

                    # ======================================
                    # TRAINING NODES OUTSIDE BUFFER
                    # ======================================

                    train_nodes = [
                        node
                        for node in development_nodes

                        if (
                            node != held_node
                            and
                            dev_distances[
                                held_node
                            ].get(
                                node,
                                np.inf
                            ) > BUFFER_KM
                        )
                    ]


                    # Number removed besides held-out node
                    removed_extra = (
                        len(development_nodes)
                        - 1
                        - len(train_nodes)
                    )


                    if len(train_nodes) < 5:
                        continue


                    # ======================================
                    # FIT
                    # ======================================

                    model = (
                        fit_graph_laplacian_regression(
                            node_table=node_table,
                            train_nodes=train_nodes,
                            node_order=node_order,
                            L=L,
                            target=TARGET,
                            features=FEATURES,
                            lambda_lap=lambda_lap,
                            lambda_beta=lambda_beta
                        )
                    )


                    observed = (
                        development.loc[
                            development["node"]
                            == held_node,
                            TARGET
                        ]
                        .iloc[0]
                    )


                    pred = (
                        model[
                            "predictions"
                        ][held_node]
                    )


                    actual.append(
                        observed
                    )

                    predicted.append(
                        pred
                    )

                    train_sizes.append(
                        len(train_nodes)
                    )

                    removed_counts.append(
                        removed_extra
                    )


                if len(actual) < 3:
                    continue


                # ======================================
                # METRICS
                # ======================================

                rmse = np.sqrt(
                    mean_squared_error(
                        actual,
                        predicted
                    )
                )

                mae = mean_absolute_error(
                    actual,
                    predicted
                )

                r2 = r2_score(
                    actual,
                    predicted
                )


                all_buffer_results.append({
                    "buffer_km":
                        BUFFER_KM,

                    "lambda_lap":
                        lambda_lap,

                    "tau_km":
                        tau,

                    "lambda_beta":
                        lambda_beta,

                    "cv_rmse":
                        rmse,

                    "cv_mae":
                        mae,

                    "cv_r2":
                        r2,

                    "actual_std":
                        np.std(actual),

                    "prediction_std":
                        np.std(predicted),

                    "mean_training_nodes":
                        np.mean(train_sizes),

                    "mean_extra_nodes_removed":
                        np.mean(removed_counts),

                    "max_extra_nodes_removed":
                        np.max(removed_counts),

                    "n_validation_predictions":
                        len(actual)
                })


all_buffer_results = (
    pd.DataFrame(
        all_buffer_results
    )
)


# Best model for each buffer
best_by_buffer = (
    all_buffer_results
    .sort_values(
        "cv_rmse"
    )
    .groupby(
        "buffer_km",
        as_index=False
    )
    .first()
)


display(
    best_by_buffer[
        [
            "buffer_km",
            "lambda_lap",
            "tau_km",
            "lambda_beta",
            "cv_rmse",
            "cv_mae",
            "cv_r2",
            "prediction_std",
            "mean_training_nodes",
            "mean_extra_nodes_removed",
            "max_extra_nodes_removed"
        ]
    ]
)


Running buffer: 75 km

Running buffer: 100 km

Running buffer: 125 km

Running buffer: 150 km


,buffer_km,lambda_lap,tau_km,lambda_beta,cv_rmse,cv_mae,cv_r2,prediction_std,mean_training_nodes,mean_extra_nodes_removed,max_extra_nodes_removed
0,75,50,50,0.01,0.087867,0.067241,0.551706,0.131578,13.333333,0.666667,2
1,100,75,50,0.10,0.086244,0.063556,0.568118,0.132179,13.066667,0.933333,3
2,125,75,50,0.10,0.087524,0.066002,0.555197,0.134659,12.800000,1.200000,3
3,150,25,75,0.10,0.089683,0.068704,0.532986,0.129769,12.400000,1.600000,4


In [48]:
import numpy as np
import pandas as pd
import networkx as nx

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. BUILD RIVER-DISTANCE MATRIX
# ============================================================

development_nodes = (
    development["node"]
    .tolist()
)

n_dev = len(development_nodes)

D = np.zeros(
    (n_dev, n_dev)
)


for i, node_i in enumerate(
    development_nodes
):

    distances = (
        nx.single_source_dijkstra_path_length(
            G_distance,
            node_i,
            weight="length_m"
        )
    )

    for j, node_j in enumerate(
        development_nodes
    ):

        if node_j in distances:

            D[i, j] = (
                distances[node_j]
                / 1000
            )

        else:

            D[i, j] = np.inf


# If anything is disconnected, replace inf
# with a very large distance.

finite_distances = D[
    np.isfinite(D)
]

large_distance = (
    finite_distances.max()
    * 2
)

D[
    ~np.isfinite(D)
] = large_distance


print(
    "Distance matrix shape:",
    D.shape
)

print(
    "Maximum river distance:",
    D.max(),
    "km"
)

Distance matrix shape: (15, 15)
Maximum river distance: 2468.3677345371557 km


In [49]:
# ============================================================
# 2. CREATE SPATIAL BLOCKS
# ============================================================

N_BLOCKS = 4


clusterer = AgglomerativeClustering(
    n_clusters=N_BLOCKS,
    metric="precomputed",
    linkage="average"
)

block_labels = clusterer.fit_predict(
    D
)


development_blocked = (
    development.copy()
)

development_blocked[
    "spatial_block"
] = block_labels


print(
    "Stations per spatial block:"
)

print(
    development_blocked[
        "spatial_block"
    ].value_counts()
    .sort_index()
)


display(
    development_blocked[
        [
            "node",
            "stations",
            "pH",
            "spatial_block"
        ]
    ].sort_values(
        "spatial_block"
    )
)

Stations per spatial block:
spatial_block
0    2
1    2
2    5
3    6
Name: count, dtype: int64


,node,stations,pH,spatial_block
9,186559442,[USGS-09510000],8.30,0
15,203177183,[USGS-09429490],8.30,0
3,67739434,[USGS-09261000],8.40,1
6,93300040,[USGS-09315000],8.40,1
5,134075067,[USGS-09365000],8.20,2
11,133114453,[USGS-09368000],8.10,2
14,131520862,[USGS-09380000],8.00,2
16,126377584,[USGS-09379500],8.20,2
17,129078768,[USGS-09371010],8.30,2
0,97532807,[USGS-09144250],8.40,3


In [50]:
# ============================================================
# 3. SPATIAL BLOCK CROSS-VALIDATION
# ============================================================

lambda_lap_values = [
    10,
    25,
    50,
    75,
    100
]

tau_values = [
    25,
    50,
    75,
    100,
    150
]

lambda_beta_values = [
    0.01,
    0.1,
    1
]


block_results = []


for tau in tau_values:

    G_L, node_order, L = (
        build_laplacian(
            G,
            tau_km=tau
        )
    )


    for lambda_lap in lambda_lap_values:

        for lambda_beta in lambda_beta_values:

            actual = []
            predicted = []


            for block in sorted(
                development_blocked[
                    "spatial_block"
                ].unique()
            ):

                # ----------------------------------------
                # ENTIRE REGION HELD OUT
                # ----------------------------------------

                train_nodes = (
                    development_blocked.loc[
                        development_blocked[
                            "spatial_block"
                        ] != block,
                        "node"
                    ]
                    .tolist()
                )

                test_nodes = (
                    development_blocked.loc[
                        development_blocked[
                            "spatial_block"
                        ] == block,
                        "node"
                    ]
                    .tolist()
                )


                # ----------------------------------------
                # FIT
                # ----------------------------------------

                model = (
                    fit_graph_laplacian_regression(
                        node_table=
                            node_table,

                        train_nodes=
                            train_nodes,

                        node_order=
                            node_order,

                        L=L,

                        target=
                            TARGET,

                        features=
                            FEATURES,

                        lambda_lap=
                            lambda_lap,

                        lambda_beta=
                            lambda_beta
                    )
                )


                # ----------------------------------------
                # PREDICT WHOLE BLOCK
                # ----------------------------------------

                for held_node in test_nodes:

                    observed = (
                        development_blocked.loc[
                            development_blocked[
                                "node"
                            ] == held_node,
                            TARGET
                        ]
                        .iloc[0]
                    )

                    pred = (
                        model[
                            "predictions"
                        ][held_node]
                    )

                    actual.append(
                        observed
                    )

                    predicted.append(
                        pred
                    )


            # --------------------------------------------
            # METRICS
            # --------------------------------------------

            rmse = np.sqrt(
                mean_squared_error(
                    actual,
                    predicted
                )
            )

            mae = mean_absolute_error(
                actual,
                predicted
            )

            r2 = r2_score(
                actual,
                predicted
            )


            block_results.append({
                "lambda_lap":
                    lambda_lap,

                "tau_km":
                    tau,

                "lambda_beta":
                    lambda_beta,

                "cv_rmse":
                    rmse,

                "cv_mae":
                    mae,

                "cv_r2":
                    r2,

                "actual_std":
                    np.std(
                        actual
                    ),

                "prediction_std":
                    np.std(
                        predicted
                    )
            })


block_results = (
    pd.DataFrame(
        block_results
    )
    .sort_values(
        "cv_rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    block_results.head(20)
)

,lambda_lap,tau_km,lambda_beta,cv_rmse,cv_mae,cv_r2,actual_std,prediction_std
0,10,50,0.01,0.127688,0.107715,0.053301,0.131233,0.133477
1,10,50,0.10,0.127826,0.104558,0.051257,0.131233,0.111219
2,25,50,0.10,0.129226,0.108062,0.030361,0.131233,0.125183
3,10,75,0.10,0.130913,0.108884,0.004885,0.131233,0.116436
4,10,75,0.01,0.136103,0.119891,-0.075592,0.131233,0.131295
5,10,100,0.10,0.136250,0.118361,-0.077920,0.131233,0.119520
6,25,75,0.10,0.138449,0.120790,-0.112994,0.131233,0.124918
7,50,50,0.10,0.139022,0.121901,-0.122226,0.131233,0.137548
8,25,50,0.01,0.139204,0.122687,-0.125155,0.131233,0.146187
9,50,75,1.00,0.139604,0.114502,-0.131640,0.131233,0.097757


In [51]:
FEATURES = [
    "Water temperature",
    "Specific conductance",
    "DO concentration",
    "Turbidity"
]

In [52]:
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

block_comparison = []

actual_all = []
mean_pred_all = []
linear_pred_all = []

# Use the best ridge value from a small search later,
# but start with the block-model beta penalty.
LINEAR_ALPHA = 0.01


for block in sorted(
    development_blocked["spatial_block"].unique()
):

    train = development_blocked[
        development_blocked["spatial_block"] != block
    ].copy()

    test = development_blocked[
        development_blocked["spatial_block"] == block
    ].copy()


    # ==================================================
    # MEAN BASELINE
    # ==================================================

    train_mean = train[TARGET].mean()

    mean_predictions = np.full(
        len(test),
        train_mean
    )


    # ==================================================
    # LINEAR REGRESSION ONLY
    # ==================================================

    # Training-only imputation
    medians = train[FEATURES].median()

    X_train = (
        train[FEATURES]
        .fillna(medians)
    )

    X_test = (
        test[FEATURES]
        .fillna(medians)
    )


    # Training-only standardization
    means = X_train.mean()

    stds = X_train.std(
        ddof=0
    ).replace(
        0,
        1
    )


    X_train = (
        X_train - means
    ) / stds

    X_test = (
        X_test - means
    ) / stds


    model = Ridge(
        alpha=LINEAR_ALPHA
    )

    model.fit(
        X_train,
        train[TARGET]
    )

    linear_predictions = model.predict(
        X_test
    )


    # ==================================================
    # SAVE RESULTS
    # ==================================================

    actual_all.extend(
        test[TARGET].tolist()
    )

    mean_pred_all.extend(
        mean_predictions.tolist()
    )

    linear_pred_all.extend(
        linear_predictions.tolist()
    )


# ==================================================
# METRIC FUNCTION
# ==================================================

def metrics(
    y_true,
    y_pred
):

    return {
        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        ),

        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),

        "R2": r2_score(
            y_true,
            y_pred
        ),

        "prediction_std": np.std(
            y_pred
        )
    }


mean_metrics = metrics(
    actual_all,
    mean_pred_all
)

linear_metrics = metrics(
    actual_all,
    linear_pred_all
)


# Best graph + linear block model
best_graph_metrics = {
    "RMSE": 0.127688,
    "MAE": 0.107715,
    "R2": 0.053301,
    "prediction_std": 0.133477
}


comparison = pd.DataFrame(
    {
        "Mean baseline":
            mean_metrics,

        "Linear only":
            linear_metrics,

        "Linear + Laplacian":
            best_graph_metrics
    }
).T


display(comparison)

,RMSE,MAE,R2,prediction_std
Mean baseline,0.163075,0.140328,-0.544143,0.043718
Linear only,0.148239,0.119964,-0.275963,0.113139
Linear + Laplacian,0.127688,0.107715,0.053301,0.133477


In [53]:
BEST_LAMBDA_LAP = 10
BEST_TAU = 50
BEST_LAMBDA_BETA = 0.01

G_L, node_order, L = build_laplacian(
    G,
    tau_km=BEST_TAU
)

block_predictions = []

for block in sorted(
    development_blocked["spatial_block"].unique()
):

    train_nodes = (
        development_blocked.loc[
            development_blocked["spatial_block"] != block,
            "node"
        ]
        .tolist()
    )

    test_block = development_blocked[
        development_blocked["spatial_block"] == block
    ].copy()

    model = fit_graph_laplacian_regression(
        node_table=node_table,
        train_nodes=train_nodes,
        node_order=node_order,
        L=L,
        target=TARGET,
        features=FEATURES,
        lambda_lap=BEST_LAMBDA_LAP,
        lambda_beta=BEST_LAMBDA_BETA
    )

    for _, row in test_block.iterrows():

        pred = model["predictions"][
            row["node"]
        ]

        block_predictions.append({
            "block": block,
            "node": row["node"],
            "stations": row["stations"],
            "observed_pH": row["pH"],
            "predicted_pH": pred,
            "residual": row["pH"] - pred,
            "absolute_error": abs(
                row["pH"] - pred
            )
        })


block_predictions = pd.DataFrame(
    block_predictions
)

display(
    block_predictions.sort_values(
        ["block", "absolute_error"],
        ascending=[True, False]
    )
)

,block,node,stations,observed_pH,predicted_pH,residual,absolute_error
0,0,186559442,[USGS-09510000],8.30,8.459060,-0.159060,0.159060
1,0,203177183,[USGS-09429490],8.30,8.256280,0.043720,0.043720
2,1,67739434,[USGS-09261000],8.40,8.200262,0.199738,0.199738
3,1,93300040,[USGS-09315000],8.40,8.381374,0.018626,0.018626
5,2,133114453,[USGS-09368000],8.10,8.344087,-0.244087,0.244087
8,2,129078768,[USGS-09371010],8.30,8.511699,-0.211699,0.211699
4,2,134075067,[USGS-09365000],8.20,8.364740,-0.164740,0.164740
6,2,131520862,[USGS-09380000],8.00,8.086833,-0.086833,0.086833
7,2,126377584,[USGS-09379500],8.20,8.282499,-0.082499,0.082499
9,3,97532807,[USGS-09144250],8.40,8.501406,-0.101406,0.101406


In [54]:
block_summary = []

for block, df in block_predictions.groupby(
    "block"
):

    rmse = np.sqrt(
        mean_squared_error(
            df["observed_pH"],
            df["predicted_pH"]
        )
    )

    mae = mean_absolute_error(
        df["observed_pH"],
        df["predicted_pH"]
    )

    if len(df) >= 2:
        r2 = r2_score(
            df["observed_pH"],
            df["predicted_pH"]
        )
    else:
        r2 = np.nan

    block_summary.append({
        "block": block,
        "n_stations": len(df),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "observed_mean":
            df["observed_pH"].mean(),
        "predicted_mean":
            df["predicted_pH"].mean()
    })


block_summary = pd.DataFrame(
    block_summary
)

display(block_summary)

,block,n_stations,rmse,mae,r2,observed_mean,predicted_mean
0,0,2,0.116644,0.101390,0.000000,8.300000,8.357670
1,1,2,0.141849,0.109182,0.000000,8.400000,8.290818
2,2,5,0.170810,0.157972,-1.805393,8.160000,8.317972
3,3,6,0.072145,0.067453,0.619537,8.341667,8.389955


In [55]:
block_predictions[
    block_predictions["block"] == 2
][
    [
        "node",
        "stations",
        "observed_pH",
        "predicted_pH",
        "residual",
        "absolute_error"
    ]
].sort_values(
    "absolute_error",
    ascending=False
)

,node,stations,observed_pH,predicted_pH,residual,absolute_error
5,133114453,[USGS-09368000],8.1,8.344087,-0.244087,0.244087
8,129078768,[USGS-09371010],8.3,8.511699,-0.211699,0.211699
4,134075067,[USGS-09365000],8.2,8.364740,-0.164740,0.164740
6,131520862,[USGS-09380000],8.0,8.086833,-0.086833,0.086833
7,126377584,[USGS-09379500],8.2,8.282499,-0.082499,0.082499


In [56]:
BLOCK = 2

block2 = development_blocked[
    development_blocked["spatial_block"] == BLOCK
].copy()

other_blocks = development_blocked[
    development_blocked["spatial_block"] != BLOCK
].copy()


print("BLOCK 2 MEANS")
print(
    block2[
        ["pH"] + FEATURES
    ].mean()
)

print()

print("TRAINING REGION MEANS")
print(
    other_blocks[
        ["pH"] + FEATURES
    ].mean()
)

BLOCK 2 MEANS
pH                        8.160
Water temperature        10.485
Specific conductance    542.600
DO concentration          8.300
Turbidity                55.610
dtype: float64

TRAINING REGION MEANS
pH                        8.345000
Water temperature        12.847500
Specific conductance    901.150000
DO concentration          9.370000
Turbidity                33.403125
dtype: float64


In [57]:
comparison = pd.DataFrame({
    "Block 2": block2[
        ["pH"] + FEATURES
    ].mean(),

    "Other blocks": other_blocks[
        ["pH"] + FEATURES
    ].mean()
})

comparison["difference"] = (
    comparison["Block 2"]
    - comparison["Other blocks"]
)

display(comparison)

,Block 2,Other blocks,difference
pH,8.160,8.345000,-0.185000
Water temperature,10.485,12.847500,-2.362500
Specific conductance,542.600,901.150000,-358.550000
DO concentration,8.300,9.370000,-1.070000
Turbidity,55.610,33.403125,22.206875


In [58]:
BLOCK = 2

train_nodes = (
    development_blocked.loc[
        development_blocked["spatial_block"] != BLOCK,
        "node"
    ]
    .tolist()
)

test_block = development_blocked[
    development_blocked["spatial_block"] == BLOCK
].copy()


G_L, node_order, L = build_laplacian(
    G,
    tau_km=50
)

model_block2 = fit_graph_laplacian_regression(
    node_table=node_table,
    train_nodes=train_nodes,
    node_order=node_order,
    L=L,
    target=TARGET,
    features=FEATURES,
    lambda_lap=10,
    lambda_beta=0.01
)


rows = []

for _, row in test_block.iterrows():

    node = row["node"]

    pred = model_block2[
        "predictions"
    ][node]

    graph_effect = model_block2[
        "graph_effect"
    ][node]

    linear_effect = (
        pred - graph_effect
    )

    rows.append({
        "node": node,
        "observed_pH": row["pH"],
        "linear_component": linear_effect,
        "graph_component": graph_effect,
        "prediction": pred,
        "error": row["pH"] - pred
    })


block2_components = pd.DataFrame(rows)

display(block2_components)

,node,observed_pH,linear_component,graph_component,prediction,error
0,134075067,8.2,8.343799,0.020941,8.364740,-0.164740
1,133114453,8.1,8.323146,0.020941,8.344087,-0.244087
2,131520862,8.0,8.065692,0.021141,8.086833,-0.086833
3,126377584,8.2,8.261558,0.020941,8.282499,-0.082499
4,129078768,8.3,8.490758,0.020941,8.511699,-0.211699


In [59]:
# ============================================================
# 1. ADD DISCHARGE AS A NODE-LEVEL FEATURE
# ============================================================

FEATURES_FLOW = [
    "Water temperature",
    "Specific conductance",
    "DO concentration",
    "Turbidity",
    "Discharge"
]

TARGET = "pH"


# Station-level medians
station_summary_flow = (
    hourly
    .groupby("station_id")[
        [TARGET] + FEATURES_FLOW
    ]
    .median()
)


node_table_flow = node_table.copy()


def get_cluster_discharge(stations):

    # Handle a single station stored as a string
    if isinstance(stations, str):
        stations = [stations]

    values = []

    for station in stations:

        if station not in station_summary_flow.index:
            continue

        value = station_summary_flow.loc[
            station,
            "Discharge"
        ]

        if pd.notna(value):
            values.append(value)

    if len(values) == 0:
        return np.nan

    return np.median(values)


node_table_flow["Discharge"] = (
    node_table_flow["stations"]
    .apply(get_cluster_discharge)
)


print(
    "Nodes with pH:",
    node_table_flow["pH"].notna().sum()
)

print(
    "Nodes with discharge:",
    node_table_flow["Discharge"].notna().sum()
)

print(
    "Nodes with both pH + discharge:",
    node_table_flow[
        ["pH", "Discharge"]
    ]
    .dropna()
    .shape[0]
)


display(
    node_table_flow[
        [
            "node",
            "role",
            "stations",
            "pH",
            "Discharge"
        ]
    ]
)

Nodes with pH: 19
Nodes with discharge: 15
Nodes with both pH + discharge: 15


,node,role,stations,pH,Discharge
0,97532807,train,[USGS-09144250],8.40,560.0
1,96496279,train,[USGS-09180500],8.30,2940.0
2,90706215,train,[USGS-09163500],8.40,2300.0
3,67739434,train,[USGS-09261000],8.40,1815.0
4,88786999,train,[USGS-09095500],8.40,1320.0
5,134075067,test_partial,[USGS-09365000],8.20,NaN
6,93300040,train,[USGS-09315000],8.40,2260.0
7,83027786,test_full,[USGS-09071750],8.30,NaN
8,81497938,test_full,[USGS-09070500],8.40,879.0
9,186559442,test_partial,[USGS-09510000],8.30,155.0


In [60]:
# ============================================================
# 2. KEEP THE EXACT SAME SPATIAL BLOCKS
# ============================================================

discharge_map = (
    node_table_flow
    .set_index("node")["Discharge"]
)

development_blocked_flow = (
    development_blocked.copy()
)

development_blocked_flow["Discharge"] = (
    development_blocked_flow["node"]
    .map(discharge_map)
)


print("Discharge coverage by block:")

display(
    development_blocked_flow
    .groupby("spatial_block")
    .agg(
        stations=("node", "count"),
        discharge_available=(
            "Discharge",
            lambda x: x.notna().sum()
        ),
        median_discharge=(
            "Discharge",
            "median"
        )
    )
)

Discharge coverage by block:


,stations,discharge_available,median_discharge
spatial_block,,,
0,2,1,155.0
1,2,2,2037.5
2,5,4,672.5
3,6,6,1815.0


In [61]:
# ============================================================
# 3. BLOCK CV WITH DISCHARGE
# ============================================================

lambda_lap_values = [
    10,
    25,
    50,
    75,
    100
]

tau_values = [
    25,
    50,
    75,
    100,
    150
]

lambda_beta_values = [
    0.01,
    0.1,
    1
]


flow_block_results = []


for tau in tau_values:

    G_L, node_order, L = build_laplacian(
        G,
        tau_km=tau
    )


    for lambda_lap in lambda_lap_values:

        for lambda_beta in lambda_beta_values:

            actual = []
            predicted = []


            for block in sorted(
                development_blocked_flow[
                    "spatial_block"
                ].unique()
            ):

                train_nodes = (
                    development_blocked_flow.loc[
                        development_blocked_flow[
                            "spatial_block"
                        ] != block,
                        "node"
                    ]
                    .tolist()
                )

                test_nodes = (
                    development_blocked_flow.loc[
                        development_blocked_flow[
                            "spatial_block"
                        ] == block,
                        "node"
                    ]
                    .tolist()
                )


                model = fit_graph_laplacian_regression(
                    node_table=node_table_flow,
                    train_nodes=train_nodes,
                    node_order=node_order,
                    L=L,
                    target=TARGET,
                    features=FEATURES_FLOW,
                    lambda_lap=lambda_lap,
                    lambda_beta=lambda_beta
                )


                for held_node in test_nodes:

                    observed = (
                        development_blocked_flow.loc[
                            development_blocked_flow[
                                "node"
                            ] == held_node,
                            TARGET
                        ]
                        .iloc[0]
                    )

                    pred = (
                        model[
                            "predictions"
                        ][held_node]
                    )

                    actual.append(
                        observed
                    )

                    predicted.append(
                        pred
                    )


            rmse = np.sqrt(
                mean_squared_error(
                    actual,
                    predicted
                )
            )

            mae = mean_absolute_error(
                actual,
                predicted
            )

            r2 = r2_score(
                actual,
                predicted
            )


            flow_block_results.append({
                "lambda_lap":
                    lambda_lap,

                "tau_km":
                    tau,

                "lambda_beta":
                    lambda_beta,

                "cv_rmse":
                    rmse,

                "cv_mae":
                    mae,

                "cv_r2":
                    r2,

                "actual_std":
                    np.std(actual),

                "prediction_std":
                    np.std(predicted)
            })


flow_block_results = (
    pd.DataFrame(
        flow_block_results
    )
    .sort_values(
        "cv_rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    flow_block_results.head(20)
)

,lambda_lap,tau_km,lambda_beta,cv_rmse,cv_mae,cv_r2,actual_std,prediction_std
0,10,100,1.0,0.137111,0.109628,-0.091582,0.131233,0.135380
1,50,50,1.0,0.137672,0.111667,-0.100538,0.131233,0.135173
2,75,50,1.0,0.137902,0.108651,-0.104209,0.131233,0.140591
3,25,75,1.0,0.138380,0.111506,-0.111886,0.131233,0.140541
4,10,150,1.0,0.138798,0.113236,-0.118612,0.131233,0.139773
5,10,75,1.0,0.138870,0.115864,-0.119765,0.131233,0.129484
6,100,50,1.0,0.139389,0.111185,-0.128160,0.131233,0.144751
7,25,100,1.0,0.140475,0.114630,-0.145794,0.131233,0.143785
8,50,75,1.0,0.141646,0.115863,-0.164982,0.131233,0.146807
9,25,150,1.0,0.142003,0.115117,-0.170856,0.131233,0.142528


In [62]:
# ============================================================
# 4. DIRECT COMPARISON
# ============================================================

best_flow = (
    flow_block_results.iloc[0]
)

comparison = pd.DataFrame({
    "Without discharge": {
        "RMSE": 0.127688,
        "MAE": 0.107715,
        "R2": 0.053301,
        "prediction_std": 0.133477
    },

    "With discharge": {
        "RMSE": best_flow["cv_rmse"],
        "MAE": best_flow["cv_mae"],
        "R2": best_flow["cv_r2"],
        "prediction_std":
            best_flow["prediction_std"]
    }
}).T


comparison["RMSE_change_%"] = (
    (
        comparison["RMSE"]
        / 0.127688
    ) - 1
) * 100


display(comparison)


print("\nBest discharge model:")
print(
    "lambda_L =",
    best_flow["lambda_lap"]
)

print(
    "tau_km =",
    best_flow["tau_km"]
)

print(
    "lambda_beta =",
    best_flow["lambda_beta"]
)

,RMSE,MAE,R2,prediction_std,RMSE_change_%
Without discharge,0.127688,0.107715,0.053301,0.133477,0.00000
With discharge,0.137111,0.109628,-0.091582,0.135380,7.37981



Best discharge model:
lambda_L = 10.0
tau_km = 100.0
lambda_beta = 1.0


In [63]:
# ============================================================
# 5. BLOCK 2 WITH BEST DISCHARGE MODEL
# ============================================================

BEST_FLOW_LAMBDA_LAP = float(
    best_flow["lambda_lap"]
)

BEST_FLOW_TAU = float(
    best_flow["tau_km"]
)

BEST_FLOW_LAMBDA_BETA = float(
    best_flow["lambda_beta"]
)


G_L, node_order, L = build_laplacian(
    G,
    tau_km=BEST_FLOW_TAU
)


BLOCK = 2

train_nodes = (
    development_blocked_flow.loc[
        development_blocked_flow[
            "spatial_block"
        ] != BLOCK,
        "node"
    ]
    .tolist()
)


test_block = (
    development_blocked_flow[
        development_blocked_flow[
            "spatial_block"
        ] == BLOCK
    ]
    .copy()
)


model_block2_flow = (
    fit_graph_laplacian_regression(
        node_table=node_table_flow,
        train_nodes=train_nodes,
        node_order=node_order,
        L=L,
        target=TARGET,
        features=FEATURES_FLOW,
        lambda_lap=
            BEST_FLOW_LAMBDA_LAP,
        lambda_beta=
            BEST_FLOW_LAMBDA_BETA
    )
)


rows = []

for _, row in test_block.iterrows():

    node = row["node"]

    prediction = (
        model_block2_flow[
            "predictions"
        ][node]
    )

    graph_component = (
        model_block2_flow[
            "graph_effect"
        ][node]
    )

    linear_component = (
        prediction
        - graph_component
    )

    rows.append({
        "node": node,

        "observed_pH":
            row["pH"],

        "Discharge":
            row["Discharge"],

        "linear_component":
            linear_component,

        "graph_component":
            graph_component,

        "prediction":
            prediction,

        "error":
            row["pH"]
            - prediction
    })


block2_flow = pd.DataFrame(
    rows
)


display(block2_flow)


print(
    "\nBlock 2 RMSE:",
    np.sqrt(
        mean_squared_error(
            block2_flow["observed_pH"],
            block2_flow["prediction"]
        )
    )
)

print(
    "Block 2 MAE:",
    mean_absolute_error(
        block2_flow["observed_pH"],
        block2_flow["prediction"]
    )
)

print(
    "Mean error:",
    block2_flow["error"].mean()
)

,node,observed_pH,Discharge,linear_component,graph_component,prediction,error
0,134075067,8.2,NaN,8.299716,0.019814,8.319530,-0.119530
1,133114453,8.1,594.0,8.341482,0.019814,8.361296,-0.261296
2,131520862,8.0,8350.0,7.919264,0.018862,7.938126,0.061874
3,126377584,8.2,721.0,8.329327,0.019814,8.349140,-0.149140
4,129078768,8.3,624.0,8.546807,0.019814,8.566621,-0.266621



Block 2 RMSE: 0.18958942981916166
Block 2 MAE: 0.17169226201581952
Mean error: -0.14694251316219945


In [64]:
coefficients = pd.Series(
    model_block2_flow[
        "coefficients"
    ],
    name="coefficient"
)

display(
    coefficients.to_frame()
)

,coefficient
Intercept,8.350739
Water temperature,-0.015024
Specific conductance,0.031775
DO concentration,0.044825
Turbidity,0.031747
Discharge,-0.040032


In [65]:
# ============================================================
# CHECK WHAT NODE ATTRIBUTES ARE ALREADY IN THE GRAPH
# ============================================================

sample_nodes = list(G.nodes())[:10]

for node in sample_nodes:
    print(
        node,
        G.nodes[node]
    )

177615876 {'longitude': -114.13666666666667, 'latitude': 34.30416666666667, 'row': 11840, 'col': 4036, 'usgs_clusters': [], 'usgs_station_count': 0, 'support_gauges': [], 'support_gauge_count': 0}
97532807 {'longitude': -108.07833333333333, 'latitude': 38.75333333333333, 'row': 6501, 'col': 11306, 'usgs_clusters': [{'cluster_id': 2, 'anchor_station': 'USGS-09144250', 'member_stations': ['USGS-09144250'], 'member_count': 1, 'features': ['DO concentration', 'Specific conductance', 'Turbidity', 'Water temperature', 'pH'], 'feature_count': 5, 'is_full_5of5': True, 'ml_role': 'train'}], 'usgs_station_count': 1, 'support_gauges': [], 'support_gauge_count': 0}
144403337 {'longitude': -114.4075, 'latitude': 36.149166666666666, 'row': 9626, 'col': 3711, 'usgs_clusters': [], 'usgs_station_count': 0, 'support_gauges': [], 'support_gauge_count': 0}
128398218 {'longitude': -113.61749999999999, 'latitude': 37.038333333333334, 'row': 8559, 'col': 4659, 'usgs_clusters': [], 'usgs_station_count': 0, 's

In [66]:
all_node_attributes = sorted({
    key
    for _, data in G.nodes(data=True)
    for key in data.keys()
})

print("Node attributes:")
for attr in all_node_attributes:
    print(attr)

Node attributes:
col
latitude
longitude
row
support_gauge_count
support_gauges
usgs_clusters
usgs_station_count


In [67]:
all_edge_attributes = sorted({
    key
    for _, _, data in G.edges(data=True)
    for key in data.keys()
})

print("Edge attributes:")
for attr in all_edge_attributes:
    print(attr)

Edge attributes:
compressed_node_count
length_m
original_edge_count
original_path
weight


In [68]:
# ============================================================
# FIND MERIT HYDRO RASTER FILES
# ============================================================

from pathlib import Path

search_root = Path(".")

raster_files = list(
    search_root.rglob("*.tif")
) + list(
    search_root.rglob("*.tiff")
)


keywords = [
    "upa",
    "elv",
    "wth",
    "hnd",
    "dir"
]


matches = {
    key: []
    for key in keywords
}


for path in raster_files:

    name = path.name.lower()

    for key in keywords:

        if key in name:
            matches[key].append(
                str(path)
            )


for key, files in matches.items():

    print(
        f"\n{key.upper()}:"
    )

    if files:

        for file in files:
            print("  ", file)

    else:

        print("   No matching file found")


UPA:
   No matching file found

ELV:
   No matching file found

WTH:
   No matching file found

HND:
   No matching file found

DIR:
   merit_colorado_dir_native.tif


In [70]:
# ============================================================
# CHECK FOR MERIT DATA ALREADY LOADED IN MEMORY
# ============================================================

possible_keywords = [
    "upa",
    "elv",
    "elev",
    "wth",
    "width",
    "hnd",
    "hand"
]

matching_variables = []

# Make a fixed copy so globals() doesn't change during iteration
current_globals = list(globals().items())

for name, obj in current_globals:

    lower = name.lower()

    if any(
        keyword in lower
        for keyword in possible_keywords
    ):

        try:
            shape = obj.shape
        except:
            shape = None

        matching_variables.append(
            (
                name,
                type(obj).__name__,
                shape
            )
        )


print("Possible MERIT variables already in memory:\n")

for name, obj_type, shape in matching_variables:

    print(
        f"{name:35s}",
        f"type={obj_type:20s}",
        f"shape={shape}"
    )

Possible MERIT variables already in memory:



In [71]:
# ============================================================
# SEARCH PARENT PROJECT DIRECTORIES FOR RASTERS
# ============================================================

from pathlib import Path

cwd = Path.cwd()

print("Current directory:")
print(cwd)

print("\nSearching from:")
print(cwd.parent.parent)


search_root = cwd.parent.parent

extensions = [
    "*.tif",
    "*.tiff",
    "*.img",
    "*.vrt"
]


raster_files = []

for extension in extensions:
    raster_files.extend(
        search_root.rglob(extension)
    )


print(
    "\nTotal raster files found:",
    len(raster_files)
)


for path in sorted(raster_files):
    print(path)

Current directory:
/Users/andrewchincea/TDA_Lab_Practice/rivernets/making_the_graph

Searching from:
/Users/andrewchincea/TDA_Lab_Practice

Total raster files found: 2
/Users/andrewchincea/TDA_Lab_Practice/rivernets/making_the_graph/merit_colorado_dir_native.tif
/Users/andrewchincea/TDA_Lab_Practice/rivernets/making_the_graph/merit_colorado_float_native.tif


In [72]:
# ============================================================
# INSPECT THE TWO MERIT RASTERS
# ============================================================

import rasterio
import numpy as np

raster_paths = [
    "merit_colorado_dir_native.tif",
    "merit_colorado_float_native.tif"
]

for path in raster_paths:

    print("\n" + "=" * 70)
    print(path)
    print("=" * 70)

    with rasterio.open(path) as src:

        print("Bands:", src.count)
        print("Shape:", (src.height, src.width))
        print("CRS:", src.crs)
        print("Transform:")
        print(src.transform)

        print("Data types:", src.dtypes)
        print("NoData:", src.nodata)

        print("\nDataset tags:")
        print(src.tags())

        for band in range(1, src.count + 1):

            print(f"\nBand {band}")

            print(
                "Description:",
                src.descriptions[band - 1]
            )

            print(
                "Band tags:",
                src.tags(band)
            )

            # Read a reduced-resolution sample so we
            # don't load the entire huge raster into RAM.
            sample = src.read(
                band,
                out_shape=(
                    1,
                    min(src.height, 1000),
                    min(src.width, 1000)
                ),
                masked=True
            )

            values = sample.compressed()

            if len(values) > 0:

                print(
                    "Sample min:",
                    np.nanmin(values)
                )

                print(
                    "Sample median:",
                    np.nanmedian(values)
                )

                print(
                    "Sample mean:",
                    np.nanmean(values)
                )

                print(
                    "Sample max:",
                    np.nanmax(values)
                )

                print(
                    "Number sampled:",
                    len(values)
                )


merit_colorado_dir_native.tif
Bands: 1
Shape: (15806, 15001)
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Transform:
| 0.00, 0.00,-117.50|
| 0.00,-0.00, 44.17|
| 0.00, 0.00, 1.00|
Data types: ('uint8',)
NoData: None

Dataset tags:
{'AREA_OR_POINT': 'Area'}

Band 1
Description: dir
Band tags: {}
Sample min: 0
Sample median: 8.0
Sample mean: 35.561467
Sample max: 128
Number sampled: 1000000

merit_colorado_float_native.tif
Bands: 4
Shape: (15806, 15001)
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Transform:
| 0.00, 0.00,-117.50|
| 0.00,-0.00, 44.17|
| 0.00, 0.00, 1.00|
Data types: ('float32', 'float32', 'float32', 'float32')
NoDat

In [73]:
# ============================================================
# CHECK WHETHER GRAPH ROW/COL FIT THE FLOAT RASTER
# ============================================================

with rasterio.open(
    "merit_colorado_float_native.tif"
) as src:

    graph_rows = [
        data["row"]
        for _, data in G.nodes(data=True)
        if "row" in data
    ]

    graph_cols = [
        data["col"]
        for _, data in G.nodes(data=True)
        if "col" in data
    ]

    print("Raster:")
    print(
        src.height,
        "rows x",
        src.width,
        "cols"
    )

    print("\nGraph row range:")
    print(
        min(graph_rows),
        "to",
        max(graph_rows)
    )

    print("\nGraph col range:")
    print(
        min(graph_cols),
        "to",
        max(graph_cols)
    )

    print(
        "\nAll graph indices inside raster:",
        (
            max(graph_rows) < src.height
            and
            max(graph_cols) < src.width
        )
    )

Raster:
15806 rows x 15001 cols

Graph row range:
2416 to 14824

Graph col range:
3118 to 12527

All graph indices inside raster: True


In [74]:
# ============================================================
# ATTACH MERIT HYDRO FEATURES TO EVERY GRAPH NODE
# ============================================================

import rasterio
import numpy as np


FLOAT_RASTER = "merit_colorado_float_native.tif"


with rasterio.open(FLOAT_RASTER) as src:

    for node, data in G.nodes(data=True):

        row = int(data["row"])
        col = int(data["col"])

        # Read one raster cell from all 4 bands
        values = src.read(
            indexes=[1, 2, 3, 4],
            window=(
                (row, row + 1),
                (col, col + 1)
            )
        )[:, 0, 0]

        upa, elv, wth, hnd = values


        G.nodes[node]["upstream_area_km2"] = (
            float(upa)
            if np.isfinite(upa)
            else np.nan
        )

        G.nodes[node]["elevation_m"] = (
            float(elv)
            if np.isfinite(elv)
            else np.nan
        )

        G.nodes[node]["river_width_m"] = (
            float(wth)
            if np.isfinite(wth)
            else np.nan
        )

        G.nodes[node]["hand_m"] = (
            float(hnd)
            if np.isfinite(hnd)
            else np.nan
        )


print("MERIT features attached.")

MERIT features attached.


In [75]:
# ============================================================
# VERIFY
# ============================================================

check = []

for node, data in G.nodes(data=True):

    check.append({
        "node": node,
        "upstream_area_km2":
            data.get("upstream_area_km2"),
        "elevation_m":
            data.get("elevation_m"),
        "river_width_m":
            data.get("river_width_m"),
        "hand_m":
            data.get("hand_m")
    })


merit_node_features = pd.DataFrame(
    check
)


display(
    merit_node_features.describe()
)

display(
    merit_node_features.head(10)
)

,node,upstream_area_km2,elevation_m,river_width_m,hand_m
count,6.700000e+01,67.000000,67.000000,67.000000,67.0
mean,1.253365e+08,96621.522752,1156.652264,145.403391,0.0
std,4.769989e+07,154512.741111,560.800940,417.049771,0.0
min,3.625120e+07,10002.214844,0.000000,0.900000,0.0
25%,8.649204e+07,11364.895020,713.700012,17.775000,0.0
50%,1.255077e+08,22887.248047,1287.700073,51.076855,0.0
75%,1.613279e+08,70707.488281,1559.650024,84.181416,0.0
max,2.223781e+08,622761.062500,2074.400146,3202.808350,0.0


,node,upstream_area_km2,elevation_m,river_width_m,hand_m
0,177615876,454124.093750,135.699997,821.802490,0.0
1,97532807,14576.997070,1499.099976,52.452950,0.0
2,144403337,419490.468750,354.300018,3202.808350,0.0
3,128398218,10002.734375,758.600037,11.250000,0.0
4,105675404,10124.354492,1257.700073,11.250000,0.0
5,83102734,15563.591797,1745.800049,56.103382,0.0
6,113101070,196714.312500,1123.099976,390.269043,0.0
7,96331283,61982.671875,1250.500000,94.328140,0.0
8,96496279,61985.140625,1250.500000,101.386337,0.0
9,102918807,10016.073242,2074.400146,11.250000,0.0


In [76]:
import pickle


with open(
    "master_compressed_river_graph_merit.pkl",
    "wb"
) as f:

    pickle.dump(
        G,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


print(
    "Saved:",
    "master_compressed_river_graph_merit.pkl"
)

Saved: master_compressed_river_graph_merit.pkl


In [77]:
# ============================================================
# FULL GRAPH FEATURE TABLE
# ============================================================

graph_feature_rows = []

for node, data in G.nodes(data=True):

    graph_feature_rows.append({
        "node":
            node,

        "longitude":
            data.get("longitude"),

        "latitude":
            data.get("latitude"),

        "upstream_area_km2":
            data.get("upstream_area_km2"),

        "elevation_m":
            data.get("elevation_m"),

        "river_width_m":
            data.get("river_width_m"),

        "hand_m":
            data.get("hand_m")
    })


graph_features = pd.DataFrame(
    graph_feature_rows
)


display(
    graph_features.head()
)

print(
    "Graph feature rows:",
    len(graph_features)
)

print(
    "\nMissing values:"
)

print(
    graph_features.isna().sum()
)

,node,longitude,latitude,upstream_area_km2,elevation_m,river_width_m,hand_m
0,177615876,-114.136667,34.304167,454124.093750,135.699997,821.80249,0.0
1,97532807,-108.078333,38.753333,14576.997070,1499.099976,52.45295,0.0
2,144403337,-114.407500,36.149167,419490.468750,354.300018,3202.80835,0.0
3,128398218,-113.617500,37.038333,10002.734375,758.600037,11.25000,0.0
4,105675404,-110.533333,38.300833,10124.354492,1257.700073,11.25000,0.0


Graph feature rows: 67

Missing values:
node                 0
longitude            0
latitude             0
upstream_area_km2    0
elevation_m          0
river_width_m        0
hand_m               0
dtype: int64


In [78]:
# ============================================================
# MERGE GRAPH FEATURES WITH WATER-QUALITY NODE TABLE
# ============================================================

node_table_merit = (
    graph_features
    .merge(
        node_table,
        on="node",
        how="left"
    )
)


print(
    "Total graph nodes:",
    len(node_table_merit)
)

print(
    "Nodes with pH:",
    node_table_merit["pH"]
    .notna()
    .sum()
)


display(
    node_table_merit[
        [
            "node",
            "pH",
            "upstream_area_km2",
            "elevation_m",
            "river_width_m",
            "hand_m"
        ]
    ]
    .dropna(
        subset=["pH"]
    )
    .head(10)
)

Total graph nodes: 67
Nodes with pH: 19


,node,pH,upstream_area_km2,elevation_m,river_width_m,hand_m
1,97532807,8.4,14576.997070,1499.099976,52.452950,0.0
8,96496279,8.3,61985.140625,1250.500000,101.386337,0.0
15,90706215,8.4,46177.554688,1326.400024,87.327324,0.0
16,67739434,8.4,65933.164062,1456.599976,81.035507,0.0
22,88786999,8.4,20663.750000,1473.900024,63.508018,0.0
29,134075067,8.2,18612.837891,1599.000000,24.299999,0.0
33,93300040,8.4,105098.984375,1236.599976,95.448524,0.0
35,83027786,8.3,11736.036133,1765.500000,38.930367,0.0
39,81497938,8.4,11368.057617,1872.200073,42.335800,0.0
40,186559442,8.3,15053.283203,482.899994,51.299988,0.0


In [79]:
# ============================================================
# TRANSFORM SKEWED RIVER VARIABLES
# ============================================================

node_table_merit[
    "log_upstream_area"
] = np.log1p(
    node_table_merit[
        "upstream_area_km2"
    ]
)

node_table_merit[
    "log_river_width"
] = np.log1p(
    node_table_merit[
        "river_width_m"
    ]
)

In [80]:
CHEMISTRY_FEATURES = [
    "Water temperature",
    "Specific conductance",
    "DO concentration",
    "Turbidity"
]


RIVER_FEATURES = [
    "elevation_m",
    "log_upstream_area",
    "log_river_width",
    "hand_m"
]


CHEMISTRY_PLUS_RIVER = (
    CHEMISTRY_FEATURES
    + RIVER_FEATURES
)